# LPRNet 微调训练 - 最终版
**完整国标字符集（77类）+ 94×48 输入 + 权重 < 500KB**

运行前请确认：Runtime → Change runtime type → T4 GPU

预计总时间：~60 分钟

In [ ]:
# Step 1: 检查 GPU
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('显存:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('未检测到 GPU，请先切换：Runtime → Change runtime type → T4 GPU')

In [ ]:
# Step 2: 安装依赖
!pip install -q opencv-python-headless pillow numpy

In [ ]:
# Step 3: 上传并解压 lprr.zip
import os, sys

os.chdir('/content')

from google.colab import files
print('请上传 lprr.zip')
uploaded = files.upload()

os.makedirs('/content/project', exist_ok=True)
os.system('unzip -q /content/lprr.zip -d /content/project/')

sys.path.insert(0, '/content/project')
os.chdir('/content/project')

print('完成！文件列表:')
print(os.listdir('lprr/'))

In [ ]:
# Step 4: 生成合成车牌数据集（约 3 分钟）
!python -m lprr.generate_plates \
    --output_dir lprr/synthetic_data \
    --num_samples 50000 \
    --val_ratio 0.1

In [ ]:
# Step 5: 训练（约 50-60 分钟）
# freeze_layers=0：不冻结任何层，全部参数参与训练
# from_scratch：忽略旧模型，从原始权重迁移重新开始
!python -m lprr.train_finetune \
    --data_dir lprr/synthetic_data \
    --epochs 50 \
    --batch_size 256 \
    --lr 1e-3 \
    --freeze_layers 0 \
    --num_workers 2 \
    --from_scratch

In [ ]:
# Step 6: 验证
import torch, os, sys
sys.path.insert(0, '/content/project')
os.chdir('/content/project')

from lprr.LPRNet import build_lprnet, INPUT_W, INPUT_H, FULL_CLASS_NUM

model_path = 'lprr/Final_LPRNet_model_full.pth'
if os.path.exists(model_path):
    size_kb = os.path.getsize(model_path) / 1024
    print(f'模型文件: {size_kb:.0f} KB')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = build_lprnet(lpr_max_len=8, phase=False, class_num=FULL_CLASS_NUM)
    model.to(device)
    state = torch.load(model_path, map_location=device)
    state = {k: v.float() if v.dtype == torch.float16 else v for k, v in state.items()}
    model.load_state_dict(state)
    model.eval()

    dummy = torch.randn(1, 3, INPUT_H, INPUT_W).to(device)
    with torch.no_grad():
        out = model(dummy)
    print(f'推理通过，输出: {out.shape}  期望: [1, {FULL_CLASS_NUM}, 18]')
    print(f'字符集: {FULL_CLASS_NUM} 类，输入: {INPUT_W}x{INPUT_H}')
    print(f'大小: {size_kb:.0f} KB {"< 500KB" if size_kb < 500 else "超出目标"}')
else:
    print('模型文件不存在，训练未完成')

In [ ]:
# Step 7: 下载模型
from google.colab import files
import os
os.chdir('/content/project')
files.download('lprr/Final_LPRNet_model_full.pth')
print('下载完成！放到项目 lprr/ 目录下，重启应用即可。')